# 🤖 ROTBOTS — Script Generator
## From Sources to Storyboard

---

1. **Feed the Machine** — Paste URLs or text, scrape and summarize
2. **The Script Writer** — LLM generates essay with narration + visual directions
3. **Visual Translation** — Storyboard with styles and T2V prompts

**You don't need to understand the code.** Just fill in inputs and press ▶️ Play.

---

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q requests beautifulsoup4 pymupdf

import json, re, random, requests, time as _time
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Setup complete')

In [ ]:
# ============================================================
# API KEY (starts with gsk_)
# Get free: https://console.groq.com/keys
# ============================================================
GROQ_API_KEY = ''  # <-- Paste key here

LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'  # NOT here

if not GROQ_API_KEY: print('⚠️  Paste your Groq API key above!')
elif not GROQ_API_KEY.startswith('gsk_'): print('⚠️  Key should start with gsk_')
else: print(f'✅ API key configured ({LLM_MODEL})')

In [ ]:
# ============================================================
# YOUR INPUTS — Topic, Sources, and Video Length
# ============================================================

TOPIC = 'The history and ethics of AI-generated art'

SOURCES = [
    'https://en.wikipedia.org/wiki/Artificial_intelligence_art',
    # 'https://example.com/another-article',
    # 'You can also paste raw text here.',
]

TARGET_VIDEO_LENGTH = 30  # seconds total
SECONDS_PER_SCENE = 5

# ============================================================
# SESSION NAME — auto-generated from topic, or set your own
# ============================================================
SESSION_NAME = ''  # Leave empty to auto-generate from topic

# Auto-generate session name from topic
if not SESSION_NAME:
    words = re.sub(r'[^a-zA-Z0-9\s]', '', TOPIC.lower()).split()
    SESSION_NAME = '-'.join(words[:4])  # First 4 words

# Create session directory
SESSION_DIR = BASE_DIR / SESSION_NAME
SESSION_DIR.mkdir(parents=True, exist_ok=True)
(SESSION_DIR / 'videos').mkdir(exist_ok=True)
(SESSION_DIR / 'audio').mkdir(exist_ok=True)

# Save session info
session_info = {'name': SESSION_NAME, 'topic': TOPIC, 'target_length': TARGET_VIDEO_LENGTH}
with open(SESSION_DIR / 'session_info.json', 'w') as f: json.dump(session_info, f, indent=2)

# Auto-calculate structure
num_content_scenes = max(2, TARGET_VIDEO_LENGTH // SECONDS_PER_SCENE)
NUM_CHAPTERS = max(1, min(3, num_content_scenes // 2))
SECTIONS_PER_CHAPTER = max(1, num_content_scenes // NUM_CHAPTERS)
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5

print(f'📂 Session: {SESSION_NAME}')
print(f'📁 Folder: {SESSION_DIR}')
print(f'🎬 Target: {TARGET_VIDEO_LENGTH}s ({num_content_scenes} scenes × {SECONDS_PER_SCENE}s)')
print(f'   {NUM_CHAPTERS} chapters × {SECTIONS_PER_CHAPTER} sections, ~{int(WORDS_PER_SCENE)} words/scene')

# Show existing sessions
sessions = [d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d / 'session_info.json').exists()]
if len(sessions) > 1:
    print(f'\n📂 Other sessions on Drive: {sessions}')

In [ ]:
# HELPER FUNCTIONS
def progress_bar(current, total, label='', width=30):
    pct = current / total if total > 0 else 0
    filled = int(width * pct)
    bar = '█' * filled + '░' * (width - filled)
    return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{bar} {current}/{total} {label} ({pct:.0%})</div>'

def progress_html(title, current, total, label='', detail='', color='#4ecca3'):
    return (
        f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">' +
        f'<div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>' +
        progress_bar(current, total, label) +
        (f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{detail}</div>' if detail else '') +
        '</div>'
    )

def query_llm(prompt, system_prompt=None, temperature=None):
    messages = []
    if system_prompt: messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    payload = {'model': LLM_MODEL, 'messages': messages, 'temperature': temperature or LLM_TEMPERATURE, 'max_tokens': LLM_MAX_TOKENS}
    response = requests.post(GROQ_API_URL, headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'}, json=payload, timeout=120)
    response.raise_for_status()
    content = response.json()['choices'][0]['message']['content']
    if '<think>' in content and '</think>' in content: content = content.split('</think>')[-1].strip()
    return content

def parse_json_response(response):
    response = response.strip()
    response = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', response)
    if response.startswith('```'):
        lines = response.split('\n')
        response = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    response = response.strip()
    if not response.startswith('[') and not response.startswith('{'):
        for ch in ['{', '[']:
            idx = response.find(ch)
            if idx != -1: response = response[idx:]; break
    for end_char in ['}', ']']:
        last_idx = response.rfind(end_char)
        if last_idx != -1:
            truncated = response[:last_idx + 1]
            for text in [truncated, re.sub(r',\s*([}\]])', r'\1', truncated)]:
                try: return json.loads(text, strict=False)
                except json.JSONDecodeError: pass
    return json.loads(re.sub(r',\s*([}\]])', r'\1', response), strict=False)

def fetch_website_text(url, max_chars=10000):
    response = requests.get(url.strip(), headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main = None
    for sel in ['article','main','[role="main"]','.content','#content']:
        main = soup.select_one(sel)
        if main: break
    text = main.get_text(separator=' ',strip=True) if main else (soup.find('body') or soup).get_text(separator=' ',strip=True)
    return re.sub(r'\s+', ' ', text).strip()[:max_chars]

def fetch_pdf_text(source, max_chars=10000):
    import tempfile
    try: import pymupdf as fitz
    except ImportError: import fitz
    temp_file = None
    try:
        if source.startswith('http'):
            resp = requests.get(source, headers={'User-Agent':'Mozilla/5.0'}, timeout=60); resp.raise_for_status()
            temp_file = tempfile.NamedTemporaryFile(suffix='.pdf', delete=False); temp_file.write(resp.content); temp_file.close()
            pdf_path = temp_file.name
        else: pdf_path = source
        doc = fitz.open(pdf_path); parts = []; total = 0
        for page in doc: t = page.get_text(); parts.append(t); total += len(t);
        doc.close()
        return re.sub(r'\n{3,}','\n\n','\n'.join(parts))[:max_chars]
    finally:
        if temp_file: import os; os.unlink(temp_file.name)

print('✅ Helpers loaded')

---
## 📥 Station 1: Feed the Machine

In [ ]:
# SCRAPE + SUMMARIZE with live progress
prog = display(HTML(''), display_id=True)
source_texts = []
total_steps = len(SOURCES) * 2  # scrape + summarize each
step = 0
t0 = _time.time()

for i, source in enumerate(SOURCES):
    prog.update(HTML(progress_html(f'📥 Scraping source {i+1}/{len(SOURCES)}', step, total_steps, 'steps', source[:60])))
    try:
        if source.startswith('http') and source.lower().endswith('.pdf'): text = fetch_pdf_text(source)
        elif source.startswith('http'): text = fetch_website_text(source)
        else: text = source
        source_texts.append({'source': source[:100], 'text': text})
    except Exception as e: pass
    step += 1

summaries = []
for i, src in enumerate(source_texts):
    prog.update(HTML(progress_html(f'🧠 Summarizing source {i+1}/{len(source_texts)}', step, total_steps, 'steps', f'LLM call... ({src["source"][:40]})')))
    summary = query_llm(f'Summarize in 2-3 short paragraphs for a video essay about "{TOPIC}":\n\n{src["text"][:6000]}', system_prompt='Be concise.')
    summaries.append({'source': src['source'], 'summary': summary})
    step += 1

with open(SESSION_DIR / 'summaries.json', 'w') as f: json.dump({'topic': TOPIC, 'sources': summaries}, f, indent=2)
prog.update(HTML(progress_html(f'✅ {len(summaries)} sources scraped & summarized in {_time.time()-t0:.1f}s', total_steps, total_steps, 'steps')))

In [ ]:
# VIEW
for i, s in enumerate(summaries):
    display(Markdown(f'### Source {i+1}: {s["source"]}'))
    display(Markdown(s['summary'] + '\n\n---'))

---
## ✍️ Station 2: The Script Writer

In [ ]:
# OUTLINE + CHAPTERS with live progress
total_llm = 1 + NUM_CHAPTERS  # 1 outline + N chapters
prog = display(HTML(''), display_id=True)
t0 = _time.time()

prog.update(HTML(progress_html('🧠 Generating essay outline...', 0, total_llm, 'LLM calls', 'Building chapter structure')))
summaries_text = '\n\n'.join([f'SOURCE: {s["source"]}\n{s["summary"]}' for s in summaries])
outline_prompt = f'Create an essay outline for a {TARGET_VIDEO_LENGTH}s video about: "{TOPIC}"\n\nRESEARCH:\n{summaries_text}\n\nExactly {NUM_CHAPTERS} chapters. JSON only:\n{{"title": "...", "thesis": "...", "chapters": [{{"chapter": 1, "title": "...", "summary": "...", "key_points": ["..."]}}]}}'
for attempt in range(3):
    try: outline = parse_json_response(query_llm(outline_prompt, 'Expert essay writer. Concise.')); break
    except Exception as e: prog.update(HTML(progress_html(f'⚠️ Retry {attempt+1}/3...', 0, total_llm, 'LLM calls', str(e)[:60], '#f39c12')))
if len(outline.get('chapters',[])) > NUM_CHAPTERS: outline['chapters'] = outline['chapters'][:NUM_CHAPTERS]

for i, ch in enumerate(outline['chapters']):
    prog.update(HTML(progress_html(f'✍️ Writing Ch {ch["chapter"]}: {ch["title"]}', i+1, total_llm, 'LLM calls', f'Chapter {i+1}/{NUM_CHAPTERS}')))
    ch_sys = f'Scriptwriter for a {TARGET_VIDEO_LENGTH}s video. Each section narration: MAX {int(WORDS_PER_SCENE)} words (1-2 sentences). Like a TikTok voiceover.'
    ch_prompt = f'Write {SECTIONS_PER_CHAPTER} sections. MAX {int(WORDS_PER_SCENE)} words narration each.\nTOPIC: {TOPIC}\nCHAPTER: {ch["title"]}\nJSON: [{{"section": 1, "narration": "MAX {int(WORDS_PER_SCENE)} WORDS", "visual_direction": "...", "mood": "..."}}]'
    try:
        sections = parse_json_response(query_llm(ch_prompt, ch_sys))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except: sections = [{'section': 1, 'narration': ch['title'], 'visual_direction': ch.get('summary',''), 'mood': 'neutral'}]
    sections = sections[:SECTIONS_PER_CHAPTER]
    for s in sections:
        words = s.get('narration','').split()
        if len(words) > int(WORDS_PER_SCENE * 1.5): s['narration'] = ' '.join(words[:int(WORDS_PER_SCENE)]) + '.'
    outline['chapters'][i]['sections'] = sections

outline['credits'] = {'sources': [s['source'] for s in summaries]}
with open(SESSION_DIR / 'essay_script.json', 'w') as f: json.dump(outline, f, indent=2)
tw = sum(len(s.get('narration','').split()) for c in outline['chapters'] for s in c.get('sections',[]))
prog.update(HTML(progress_html(f'✅ "{outline.get("title","")}" — {tw} words ≈ {tw/2.5:.0f}s narration ({_time.time()-t0:.1f}s)', total_llm, total_llm, 'LLM calls')))

In [ ]:
# VIEW
display(Markdown(f'# {outline.get("title","")}\n*{outline.get("thesis","")}*\n\n---'))
for ch in outline['chapters']:
    display(Markdown(f'## Ch {ch["chapter"]}: {ch["title"]}'))
    for s in ch.get('sections',[]):
        wc=len(s.get('narration','').split())
        display(Markdown(f'**Sec {s.get("section","?")}** *{s.get("mood","?")}* ({wc}w≈{wc/2.5:.0f}s)\n\n> 🎙️ {s.get("narration","")}\n\n> 🎬 {s.get("visual_direction","")}\n\n---'))

---
## 🎬 Station 3: Visual Translation

In [ ]:
# STYLE ARC
STYLE_ARCS = {
    'calm_to_dramatic': {'description': 'Calm → intense', 'early': ['documentary','nature'], 'middle': ['news_report','sports_commentary'], 'late': ['action_movie','horror']},
    'documentary_journey': {'description': 'Documentary, increasing energy', 'early': ['documentary'], 'middle': ['nature','news_report'], 'late': ['action_movie','sports_commentary']},
    'chaos_arc': {'description': 'Brainrot chaos', 'early': ['classic_brainrot','zoomer_brainrot'], 'middle': ['surreal_brainrot','hyperpop_brainrot'], 'late': ['fever_dream_brainrot','cursed_brainrot']},
}
STYLES = {
    'documentary':{'vk':'cinematic, professional lighting','ak':'ambient sounds'},
    'nature':{'vk':'wide nature shots, golden hour','ak':'nature sounds, wind'},
    'news_report':{'vk':'news studio, professional','ak':'news audio, serious'},
    'sports_commentary':{'vk':'dynamic angles, slow motion','ak':'crowd, energetic'},
    'action_movie':{'vk':'dynamic movement, dramatic lighting','ak':'orchestral hits'},
    'horror':{'vk':'dark lighting, shadows, fog','ak':'creepy ambience'},
    'classic_brainrot':{'vk':'chaotic, deep fried','ak':'bass boosted'},
    'surreal_brainrot':{'vk':'dreamlike, impossible geometry','ak':'slowed reverb'},
    'zoomer_brainrot':{'vk':'meme aesthetic, ironic','ak':'TikTok sounds'},
    'hyperpop_brainrot':{'vk':'maximalist, Y2K','ak':'hyperpop beats'},
    'fever_dream_brainrot':{'vk':'hallucinatory, warped','ak':'echoing'},
    'cursed_brainrot':{'vk':'unsettling, jpeg artifacts','ak':'distorted'},
}

CHOSEN_ARC = 'calm_to_dramatic'
arc = STYLE_ARCS[CHOSEN_ARC]
print(f'🎨 {CHOSEN_ARC}: {arc["description"]}')

In [ ]:
# STORYBOARD + STYLES
scenes = []; sn = 1
scenes.append({'scene':sn,'scene_type':'title_card','title':outline.get('title',''),'description':outline.get('thesis','')}); sn+=1
for ch in outline.get('chapters',[]):
    for sec in ch.get('sections',[]):
        scenes.append({'scene':sn,'scene_type':'content','title':f"{ch['title']} - Part {sec.get('section',1)}",'narration':sec.get('narration',''),'visual_direction':sec.get('visual_direction',''),'mood':sec.get('mood',''),'chapter':ch.get('chapter',0)}); sn+=1
scenes.append({'scene':sn,'scene_type':'credits','title':'Credits','description':','.join(outline.get('credits',{}).get('sources',[]))})

cs=[s for s in scenes if s['scene_type']=='content']; total=len(cs)
ee=max(1,int(total*0.25)); ls=max(ee+1,int(total*0.75)); last=None
for i,sc in enumerate(cs):
    pool=arc.get('early' if i<ee else 'late' if i>=ls else 'middle',['documentary'])
    av=[s for s in pool if s!=last] or pool; st=random.choice(av)
    sc['assigned_style']=st; sc['visual_keywords']=STYLES.get(st,{}).get('vk',''); sc['audio_keywords']=STYLES.get(st,{}).get('ak',''); last=st

with open(SESSION_DIR/'storyboard.json','w') as f: json.dump(scenes,f,indent=2)
for s in scenes:
    tag=f' [{s.get("assigned_style","")}]' if s.get('assigned_style') else ''
    print(f'   Scene {s["scene"]}: [{s["scene_type"]}] {s["title"]}{tag}')

In [ ]:
# T2V PROMPTS with live progress
content_scenes = [sc for sc in scenes if sc['scene_type']=='content']
OPENINGS = ['Start with SHOT TYPE','Start with ACTION','Start with ENVIRONMENT','Start with LIGHTING','Start with CAMERA']
prompts = []
prog = display(HTML(''), display_id=True)
t0 = _time.time()

for idx, sc in enumerate(content_scenes):
    st=sc.get('assigned_style','none')
    prog.update(HTML(progress_html(f'🎥 Scene {sc["scene"]}: {sc["title"]}', idx, len(content_scenes), 'prompts', f'Style: {st} • Calling LLM...')))
    si=f'\nStyle:{st} Visual:{sc.get("visual_keywords","")} Audio:{sc.get("audio_keywords","")}' if st!='none' else ''
    sys_p=f'T2V prompt expert. ONE paragraph, 3-5 sentences for {SECONDS_PER_SCENE}s clip.{si} No text/writing.'
    user_p=f'T2V prompt for {SECONDS_PER_SCENE}s:\nVisual: {sc.get("visual_direction","")}\nMood: {sc.get("mood","")}\n{random.choice(OPENINGS)}\nOutput ONLY prompt.'
    t2v=query_llm(user_p,sys_p).strip().strip('"')
    prompts.append({'scene':sc['scene'],'title':sc['title'],'t2v_prompt':t2v,'style':st,'narration':sc.get('narration',''),'duration':SECONDS_PER_SCENE})

with open(SESSION_DIR/'prompts.json','w') as f: json.dump(prompts,f,indent=2)
prog.update(HTML(progress_html(f'✅ {len(prompts)} prompts generated in {_time.time()-t0:.1f}s', len(content_scenes), len(content_scenes), 'prompts')))

In [ ]:
# VIEW
display(Markdown(f'# 🎬 {len(prompts)} scenes × ~{SECONDS_PER_SCENE}s = ~{len(prompts)*SECONDS_PER_SCENE}s\n\n---'))
for p in prompts:
    wc=len(p['narration'].split())
    display(Markdown(f'### Scene {p["scene"]}: {p["title"]}\n`{p["style"]}` | {p["duration"]}s | {wc}w\n\n> 🎙️ {p["narration"]}\n\n> 🎥 {p["t2v_prompt"]}\n\n---'))
print(f'📂 Session "{SESSION_NAME}" saved to Google Drive')

---
*ROTBOTS Workshop — LI-MA 2026*